# QC BIDS Events for OpenNeuro Feedback

This notebook only checks the current BIDS `events.tsv` and `events.json` files. It does not modify the dataset.

It answers the testing questions before any processing step:

1. Is `raw_trial_type` redundant or does it preserve old marker formatting?
2. Is raw trigger `value` the same as WAV `stimulus_id`?
3. Would the `duration` column change if it were replaced by WAV durations?
4. What `events.json` metadata should be added for `stimulus_id`?


In [1]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import wave

import pandas as pd

BIDS_ROOT = Path('/Users/yanyuwoo/Data/bids')
STIMULI_DIR = BIDS_ROOT / 'stimuli'
PROJECT_DATA_ROOT = Path('/Users/yanyuwoo/Data/Alice Comprehension')
QC_DIR = PROJECT_DATA_ROOT / 'qc' / 'openneuro_events_feedback'
BACKUP_ROOT = PROJECT_DATA_ROOT / 'intermediate' / 'openneuro_events_feedback_backups'

# Keep False until the dry-run reports look correct.

QC_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

print(f'BIDS root: {BIDS_ROOT}')
print(f'Stimuli dir: {STIMULI_DIR}')


BIDS root: /Users/yanyuwoo/Data/bids
Stimuli dir: /Users/yanyuwoo/Data/bids/stimuli


## Gather Files and WAV Durations

Durations are read from the actual WAV files so the notebook does not depend on hard-coded segment lengths.

In [2]:
def subject_from_events_path(path):
    return path.parts[-3]


def read_wav_duration(path):
    with wave.open(str(path), 'rb') as wav:
        return wav.getnframes() / wav.getframerate()


events_paths = sorted(
    BIDS_ROOT.glob('sub-*/eeg/sub-*_task-alice_events.tsv'),
    key=lambda p: int(subject_from_events_path(p).split('-')[1]),
)
events_json_paths = [path.with_suffix('.json') for path in events_paths]

wav_durations = {
    int(path.stem): read_wav_duration(path)
    for path in sorted(STIMULI_DIR.glob('*.wav'), key=lambda p: int(p.stem))
}

duration_table = pd.DataFrame(
    [{'stimulus_id': key, 'wav_duration': value} for key, value in wav_durations.items()]
)

print(f'Events files: {len(events_paths)}')
print(f'WAV files: {len(wav_durations)}')
display(duration_table)

Events files: 49
WAV files: 12


,stimulus_id,wav_duration
0,1,57.540612
1,2,60.845193
2,3,63.259433
3,4,69.988571
4,5,66.272540
5,6,63.777551
6,7,62.896848
7,8,57.310612
8,9,57.226145
9,10,61.269660


## 1. Inspect `raw_trial_type`

This cell reports whether `raw_trial_type` duplicates `trial_type`. If the column is removed from the public BIDS events files later, this report and the processing backups preserve the provenance.

In [3]:
raw_trial_rows = []

for path in events_paths:
    subject = subject_from_events_path(path)
    df = pd.read_csv(path, sep='\t')
    has_raw = 'raw_trial_type' in df.columns
    if has_raw:
        identical = df['raw_trial_type'].astype(str).equals(df['trial_type'].astype(str))
        raw = df['raw_trial_type'].astype(str)
        trial = df['trial_type'].astype(str)
        n_different = int((df['raw_trial_type'].astype(str) != df['trial_type'].astype(str)).sum())
        examples = df.loc[df['raw_trial_type'].astype(str) != df['trial_type'].astype(str), ['trial_type', 'raw_trial_type']].head(3).to_dict('records')
    else:
        identical = None
        n_different = 0
        examples = []

    raw_trial_rows.append({
        'subject': subject,
        # 'has_raw_trial_type': has_raw,
        'is_identical': identical,
        'raw': raw,
        'trial': trial,
        'n_different_rows': n_different,
        'different_examples': json.dumps(examples),
    })

raw_trial_report = pd.DataFrame(raw_trial_rows)
raw_trial_report_path = QC_DIR / 'raw_trial_type_report.csv'
raw_trial_report.to_csv(raw_trial_report_path, index=False)

print(f'Wrote report: {raw_trial_report_path}')
display(raw_trial_report['is_identical'].value_counts(dropna=False).rename('n_subjects'))
# display(raw_trial_report.head())
# display(raw_trial_report)
display(raw_trial_report[raw_trial_report["is_identical"] == False])

# An example of a subject with a mismatch.
# Difference mainly due to the format of trial type column
subject = "sub-21"
path = BIDS_ROOT / subject / "eeg" / f"{subject}_task-alice_events.tsv"
df = pd.read_csv(path, sep="\t")

display(df[["trial_type", "raw_trial_type", "stimulus_id", "value"]])


Wrote report: /Users/yanyuwoo/Data/Alice Comprehension/qc/openneuro_events_feedback/raw_trial_type_report.csv


is_identical
False    29
True     20
Name: n_subjects, dtype: int64

,subject,is_identical,raw,trial,n_different_rows,different_examples
20,sub-21,False,0 Stimulus/S 1 1 Stimulus/S 2 2 ...,0 Stimulus/1 1 Stimulus/2 2 Sti...,12,"[{""trial_type"": ""Stimulus/1"", ""raw_trial_type""..."
21,sub-22,False,0 Stimulus/S 1 1 Stimulus/S 2 2 ...,0 Stimulus/1 1 Stimulus/2 2 Sti...,12,"[{""trial_type"": ""Stimulus/1"", ""raw_trial_type""..."
22,sub-23,False,0 Stimulus/S 1 1 Stimulus/S 2 2 ...,0 Stimulus/1 1 Stimulus/2 2 Sti...,12,"[{""trial_type"": ""Stimulus/1"", ""raw_trial_type""..."
23,sub-24,False,0 Stimulus/S 2 1 Stimulus/S 3 2 ...,0 Stimulus/2 1 Stimulus/3 2 Sti...,11,"[{""trial_type"": ""Stimulus/2"", ""raw_trial_type""..."
24,sub-25,False,0 Stimulus/S 1 1 Stimulus/S 2 2 ...,0 Stimulus/1 1 Stimulus/2 2 Sti...,12,"[{""trial_type"": ""Stimulus/1"", ""raw_trial_type""..."
25,sub-26,False,0 Stimulus/S 2 1 Stimulus/S 3 2 ...,0 Stimulus/2 1 Stimulus/3 2 Sti...,11,"[{""trial_type"": ""Stimulus/2"", ""raw_trial_type""..."
26,sub-27,False,0 Stimulus/S 2 1 Stimulus/S 3 2 ...,0 Stimulus/2 1 Stimulus/3 2 Sti...,11,"[{""trial_type"": ""Stimulus/2"", ""raw_trial_type""..."
27,sub-28,False,0 Stimulus/S 1 1 Stimulus/S 2 2 ...,0 Stimulus/1 1 Stimulus/2 2 Sti...,12,"[{""trial_type"": ""Stimulus/1"", ""raw_trial_type""..."
28,sub-29,False,0 Stimulus/S 2 1 Stimulus/S 3 2 ...,0 Stimulus/2 1 Stimulus/3 2 Sti...,11,"[{""trial_type"": ""Stimulus/2"", ""raw_trial_type""..."
29,sub-30,False,0 Stimulus/S 2 1 Stimulus/S 3 2 ...,0 Stimulus/2 1 Stimulus/3 2 Sti...,11,"[{""trial_type"": ""Stimulus/2"", ""raw_trial_type""..."


,trial_type,raw_trial_type,stimulus_id,value
0,Stimulus/1,Stimulus/S 1,1,1
1,Stimulus/2,Stimulus/S 2,2,2
2,Stimulus/3,Stimulus/S 3,3,3
3,Stimulus/4,Stimulus/S 4,4,4
4,Stimulus/5,Stimulus/S 5,5,5
5,Stimulus/6,Stimulus/S 6,6,6
6,Stimulus/7,Stimulus/S 7,7,7
7,Stimulus/8,Stimulus/S 8,8,8
8,Stimulus/9,Stimulus/S 9,9,9
9,Stimulus/10,Stimulus/S 10,10,10


## 2. Check `value` vs `stimulus_id`

`value` is the raw trigger code. `stimulus_id` should refer to the WAV segment identity. This report makes the distinction explicit instead of assuming they are the same.

In [5]:
value_rows = []

for path in events_paths:
    subject = subject_from_events_path(path)
    df = pd.read_csv(path, sep='\t')
    if 'stimulus_id' not in df.columns:
        value_rows.append({'subject': subject, 'status': 'missing_stimulus_id'})
        continue

    for _, row in df.iterrows():
        value_rows.append({
            'subject': subject,
            'onset': row.get('onset'),
            'trial_type': row.get('trial_type'),
            'value': row.get('value'),
            'stimulus_id': row.get('stimulus_id'),
            'value_equals_stimulus_id': row.get('value') == row.get('stimulus_id'),
            'status': 'ok',
        })

value_stimulus_report = pd.DataFrame(value_rows)
value_stimulus_report_path = QC_DIR / 'value_vs_stimulus_id_report.csv'
value_stimulus_report.to_csv(value_stimulus_report_path, index=False)

print(f'Wrote report: {value_stimulus_report_path}')
display(value_stimulus_report['value_equals_stimulus_id'].value_counts(dropna=False).rename('n_events'))
display(value_stimulus_report.head(20))

Wrote report: /Users/yanyuwoo/Data/Alice Comprehension/qc/openneuro_events_feedback/value_vs_stimulus_id_report.csv


value_equals_stimulus_id
False    330
True     247
Name: n_events, dtype: int64

,subject,onset,trial_type,value,stimulus_id,value_equals_stimulus_id,status
0,sub-01,3.664,Stimulus/1,1,1,True,ok
1,sub-01,61.292,Stimulus/2,5,2,False,ok
2,sub-01,122.188,Stimulus/3,6,3,False,ok
3,sub-01,185.500,Stimulus/4,7,4,False,ok
4,sub-01,255.544,Stimulus/5,8,5,False,ok
5,sub-01,321.872,Stimulus/6,9,6,False,ok
6,sub-01,385.700,Stimulus/7,10,7,False,ok
7,sub-01,448.660,Stimulus/8,11,8,False,ok
8,sub-01,506.022,Stimulus/9,12,9,False,ok
9,sub-01,563.302,Stimulus/10,2,10,False,ok


## 3. Preview WAV Duration Replacement

The current `duration` column appears to describe the trigger pulse. For OpenNeuro, the event duration should describe the duration of the stimulus event. This cell previews replacing `duration` with the corresponding WAV duration.

In [6]:
duration_rows = []

for path in events_paths:
    subject = subject_from_events_path(path)
    df = pd.read_csv(path, sep='\t')
    if 'stimulus_id' not in df.columns:
        duration_rows.append({'subject': subject, 'status': 'missing_stimulus_id'})
        continue

    expected_duration = pd.to_numeric(
        df['stimulus_id'].map(lambda value: wav_durations.get(int(value)) if pd.notna(value) else None),
        errors='coerce',
    )
    observed_duration = pd.to_numeric(df['duration'], errors='coerce')
    changed = (observed_duration.round(6) != expected_duration.round(6)).fillna(True)
    duration_rows.append({
        'subject': subject,
        'n_rows': len(df),
        'n_duration_changes': int(changed.sum()),
        'old_duration_examples': ';'.join(map(str, sorted(df['duration'].dropna().unique())[:5])),
        'new_duration_examples': ';'.join(map(str, sorted(expected_duration.dropna().round(6).unique())[:5])),
        'status': 'ok',
    })

duration_report = pd.DataFrame(duration_rows)
duration_report_path = QC_DIR / 'duration_replacement_report.csv'
duration_report.to_csv(duration_report_path, index=False)

print(f'Wrote report: {duration_report_path}')
display(duration_report.head())

Wrote report: /Users/yanyuwoo/Data/Alice Comprehension/qc/openneuro_events_feedback/duration_replacement_report.csv


,subject,n_rows,n_duration_changes,old_duration_examples,new_duration_examples,status
0,sub-01,12,12,0.002,46.983061;56.170181;57.226145;57.310612;57.540612,ok
1,sub-02,11,11,0.002,46.983061;56.170181;57.226145;57.310612;60.845193,ok
2,sub-03,12,12,0.002,46.983061;56.170181;57.226145;57.310612;57.540612,ok
3,sub-04,12,12,0.002,46.983061;56.170181;57.226145;57.310612;57.540612,ok
4,sub-05,12,12,0.002,46.983061;56.170181;57.226145;57.310612;57.540612,ok


## 4. Prepare `events.json` Metadata

This adds documentation for `stimulus_id` and clarifies that `value` is the raw trigger code, not the WAV segment identity.

In [7]:
STIMULUS_ID_METADATA = {
    'Description': 'Identifier of the Alice audio segment presented at this event. This refers to the WAV stimulus file in the stimuli directory, not the raw trigger code.',
    'Levels': {str(i): f'Alice chapter-one audio segment {i}; stimulus file {i}.wav' for i in sorted(wav_durations)},
}

EVENTS_JSON_UPDATES = {
    'duration': {
        'Description': 'Duration of the presented audio stimulus in seconds, measured from the corresponding WAV file.',
        'Units': 's',
    },
    'value': {
        'Description': 'Raw trigger code associated with the event. This is preserved from the recording system and should not be assumed to equal stimulus_id.',
    },
    'trial_type': {
        'Description': 'Standardized stimulus label derived from the event marker, using Stimulus/1 through Stimulus/12.',
    },
    'stimulus_id': STIMULUS_ID_METADATA,
}

example_json_path = events_json_paths[0]
with example_json_path.open(encoding='utf-8') as f:
    example_json = json.load(f)
preview_json = {**example_json, **EVENTS_JSON_UPDATES}

print(f'Example JSON path: {example_json_path}')
print(json.dumps(preview_json, indent=2)[:2000])

Example JSON path: /Users/yanyuwoo/Data/bids/sub-01/eeg/sub-01_task-alice_events.json
{
  "onset": {
    "Description": "Onset (in seconds) of the event from the beginning of the first datapoint. Negative onsets account for events before the first stored data point.",
    "Units": "s"
  },
  "duration": {
    "Description": "Duration of the presented audio stimulus in seconds, measured from the corresponding WAV file.",
    "Units": "s"
  },
  "sample": {
    "Description": "The event onset time in number of sampling points.First sample is 0."
  },
  "value": {
    "Description": "Raw trigger code associated with the event. This is preserved from the recording system and should not be assumed to equal stimulus_id."
  },
  "trial_type": {
    "Description": "Standardized stimulus label derived from the event marker, using Stimulus/1 through Stimulus/12."
  },
  "stimulus_id": {
    "Description": "Identifier of the Alice audio segment presented at this event. This refers to the WAV stim

## Summary Tables

These summaries are meant to make the conclusion explicit after running the QC cells above.

In [8]:
print('raw_trial_type identical to trial_type:')
if 'raw_trial_report' in globals():
    col = 'raw_trial_type_identical_to_trial_type' if 'raw_trial_type_identical_to_trial_type' in raw_trial_report.columns else 'is_identical'
    display(raw_trial_report[col].value_counts(dropna=False).rename('n_subjects'))

print('value equals stimulus_id:')
if 'value_stimulus_report' in globals() and 'value_equals_stimulus_id' in value_stimulus_report.columns:
    display(value_stimulus_report['value_equals_stimulus_id'].value_counts(dropna=False).rename('n_events'))

print('duration changes needed:')
if 'duration_report' in globals() and 'n_duration_changes' in duration_report.columns:
    print(f"Subjects with any duration change: {(duration_report['n_duration_changes'] > 0).sum()} / {len(duration_report)}")
    print(f"Total event rows with duration change: {duration_report['n_duration_changes'].sum()}")


raw_trial_type identical to trial_type:


is_identical
False    29
True     20
Name: n_subjects, dtype: int64

value equals stimulus_id:


value_equals_stimulus_id
False    330
True     247
Name: n_events, dtype: int64

duration changes needed:
Subjects with any duration change: 49 / 49
Total event rows with duration change: 577


## Important Boundary

This notebook checks event-table consistency and metadata. It does not verify whether `stimulus_id` is acoustically correct. Use `02_check_stimulus_id_audio_alignment.ipynb` for audio-track verification before applying event cleanup.